# 05 — Sensitivity and efficiency

One-factor-at-a-time sensitivity around the main recipe. The notebook varies the only loss weight $\lambda$, the batch on which $H_0$ is estimated, and the size of the fixed calibration subset reused by every epoch-wise Procrustes refit. It also reads measured step time and peak memory from the same runs.


In [2]:
# 1. Cấu hình
from pathlib import Path
REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO = True
INSTALL_REQUIREMENTS = True
PAIR = "qwen3_0.6b_to_minilm_h384"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"sensitivity_{PAIR}_v1"
SEEDS = [42, 43, 44]
DEFAULT_LAMBDA, DEFAULT_BATCH = 0.75, 128
DEFAULT_GAUGE_SAMPLES, DEFAULT_REFIT = 16384, 1
LAMBDA_VALUES = [0.0, 0.3, 0.5, 0.75, 1.0]
BATCH_VALUES = [16, 64, 128, 256]
GAUGE_SAMPLE_VALUES = [2048, 16384, 32768, 65536]
EPOCHS, LR = 5, 7e-5
EXECUTE = False
CUDA_VISIBLE_DEVICES = "0"


In [3]:
# 2. Clone/fetch repo, dependencies, imports và output
import shlex, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
if AUTO_PULL_REPO:
    dirty = subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"], check=True, capture_output=True, text=True).stdout.strip()
    if dirty:
        print("[git] Bỏ qua pull vì repo có tracked changes.")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
git_head = subprocess.run(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
from _analysis_common import PAIRS, collect_jobs, geoode_command, read_jsonl, run_jobs, set_paper_style
PAIR_CONFIG = PAIRS[PAIR]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
RUN_ROOT = PROJECT_DIR / "runs" / RUN_NAME
CACHE_DIR = PROJECT_DIR / "runs" / "teacher_cache"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
set_paper_style()
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Output: {RUN_ROOT}")


ModuleNotFoundError: No module named '_analysis_common'

In [ ]:
# 3. OFAT plan; the default configuration is run once and reused in every panel
specs = [{"name": "default", "sweep": "default", "value": np.nan, "lambda": DEFAULT_LAMBDA, "batch": DEFAULT_BATCH, "gauge_samples": DEFAULT_GAUGE_SAMPLES, "refit": DEFAULT_REFIT}]
specs += [{"name": f"lambda_{value:g}", "sweep": "lambda", "value": value, "lambda": value, "batch": DEFAULT_BATCH, "gauge_samples": DEFAULT_GAUGE_SAMPLES, "refit": DEFAULT_REFIT} for value in LAMBDA_VALUES]
specs += [{"name": f"batch_{value}", "sweep": "batch", "value": value, "lambda": DEFAULT_LAMBDA, "batch": value, "gauge_samples": DEFAULT_GAUGE_SAMPLES, "refit": DEFAULT_REFIT} for value in BATCH_VALUES]
specs += [{"name": f"gauge_samples_{value}", "sweep": "gauge_samples", "value": value, "lambda": DEFAULT_LAMBDA, "batch": DEFAULT_BATCH, "gauge_samples": value, "refit": DEFAULT_REFIT} for value in GAUGE_SAMPLE_VALUES]
jobs = []
for spec in specs:
    for seed in SEEDS:
        run_dir = RUN_ROOT / spec["name"] / f"seed_{seed}"
        extra = ["--projection_type", "pca", "--gauge_align", "--gauge_rotation", "procrustes", "--gauge_refit_every", str(spec["refit"]), "--gauge_align_samples", str(spec["gauge_samples"]), "--lambda_end", "1", "--lambda_ctr", "0", "--lambda_topo", str(spec["lambda"]), "--lambda_h1", "0", "--topo_teacher_source", "original", "--no_eval_retrieval"]
        jobs.append({"name": f"{spec['name']}/seed_{seed}", "arm": spec["name"], "sweep": spec["sweep"], "value": spec["value"], "seed": seed, "run_dir": run_dir, "command": geoode_command(PROJECT_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA, cache_dir=CACHE_DIR, run_dir=run_dir, seed=seed, batch_size=spec["batch"], epochs=EPOCHS, learning_rate=LR, extra=extra)})
print(f"Plan: {len(jobs)} jobs")
for job in jobs:
    print(shlex.join(job["command"]))
if EXECUTE:
    display(run_jobs(PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES))


In [ ]:
# 4. Join quality with measured throughput and peak memory
results = collect_jobs(jobs)
runtime_rows = []
for job in jobs:
    steps = read_jsonl(Path(job["run_dir"]) / "step_metrics.jsonl")
    metrics = read_jsonl(Path(job["run_dir"]) / "metrics.jsonl")
    timed = steps.query("step_seconds > 0") if not steps.empty else steps
    peak = np.nan
    if not metrics.empty and "train" in metrics:
        peak_values = [row.get("peak_memory_mb", np.nan) for row in metrics["train"].dropna() if isinstance(row, dict)]
        peak = np.nanmax(peak_values) if peak_values else np.nan
    runtime_rows.append({"arm": job["arm"], "seed": job["seed"], "train_seconds": timed["step_seconds"].sum() if not timed.empty else np.nan, "samples_per_second": timed["batch_size"].sum() / timed["step_seconds"].sum() if not timed.empty else np.nan, "peak_memory_mb": peak})
runtime = pd.DataFrame(runtime_rows)
results = results.merge(runtime, on=["arm", "seed"], how="left")
results.to_csv(RUN_ROOT / "sensitivity_efficiency_by_seed.csv", index=False)
done = results.query("status == 'done'").copy()
summary = done.groupby(["sweep", "value"], dropna=False).agg(avg_all=("avg_all", "mean"), score_std=("avg_all", "std"), throughput=("samples_per_second", "mean"), peak_memory_mb=("peak_memory_mb", "mean"), n=("avg_all", "count")).reset_index()
summary.to_csv(RUN_ROOT / "sensitivity_efficiency_summary.csv", index=False)
display(summary.style.format(precision=4))


In [ ]:
# 5. Appendix Figure A2 — three ordered sensitivity curves
if summary.empty:
    print("No completed sensitivity runs yet.")
else:
    default = done.query("sweep == 'default'").copy()
    curve_panels = [
        ("lambda", DEFAULT_LAMBDA, r"topology weight $\lambda$"),
        ("batch", DEFAULT_BATCH, r"$H_0$ batch size"),
        ("gauge_samples", DEFAULT_GAUGE_SAMPLES, "frozen gauge samples"),
    ]
    colors = ["#2B6CB0", "#2F855A", "#6B46C1"]
    fig, axes = plt.subplots(1, 3, figsize=(5.5, 2.15), sharey=True)
    for panel_index, (ax, (sweep, default_value, xlabel), color) in enumerate(zip(axes, curve_panels, colors)):
        part = done.query("sweep == @sweep")[["value", "avg_all"]].copy()
        base = default[["avg_all"]].copy(); base["value"] = default_value
        part = pd.concat([part, base], ignore_index=True)
        stats = part.groupby("value")["avg_all"].agg(["mean", "std"]).reset_index().sort_values("value")
        x = np.arange(len(stats)); mean = 100 * stats["mean"].to_numpy(); sd = 100 * stats["std"].fillna(0).to_numpy()
        ax.plot(x, mean, marker="o", ms=3.5, color=color)
        ax.fill_between(x, mean - sd, mean + sd, color=color, alpha=.18, linewidth=0)
        default_x = int(np.flatnonzero(np.isclose(stats["value"], default_value))[0])
        ax.axvline(default_x, color="#DD6B20", ls="--", lw=1)
        labels = ([f"{value:g}" for value in stats["value"]] if sweep == "lambda" else [f"{int(value / 1024)}k" for value in stats["value"]] if sweep == "gauge_samples" else [f"{int(value)}" for value in stats["value"]])
        ax.set(xticks=x, xticklabels=labels, xlabel=xlabel, title=f"({chr(97 + panel_index)})")
        ax.grid(axis="y", color="#E5E7EB", lw=.7); ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel("Final AVG ×100")
    fig.text(.02, .01, "Mean ± sample SD over 3 seeds; dashed line marks the default.", fontsize=6.5, color="#6B7280")
    fig.tight_layout(rect=(0, .06, 1, 1), w_pad=.8)
    fig.savefig(RUN_ROOT / "figure_A2_sensitivity.pdf", bbox_inches="tight")
    fig.savefig(RUN_ROOT / "figure_A2_sensitivity.png", dpi=300, bbox_inches="tight")
    plt.show()
